# Travel Agent - Trip Planning System

This notebook contains a comprehensive travel planning system using MCP (Model Context Protocol) servers with AI agents for trip planning, flight finding, and activity planning.

In [15]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from pydantic import BaseModel
from typing import List, Dict
from datetime import datetime, timedelta

load_dotenv(override=True)

True

In [16]:
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}
]
mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in mcp_server_params]

In [17]:
# Trip Configuration
start_date = "2025-10-01"
end_date = "2025-10-10"
start_destination = "Toronto"
destinations = ["Nova Scotia"]
additional_info = ""

## Data Models

In [18]:
class Destination(BaseModel):
    city: str
    days: int
    

class TripDividerOutput(BaseModel):
    Destinations: List[Destination]


class FlightFinderOutput(BaseModel):
    Airline: str
    Departure: str
    Arrival: str
    Departure_Time: str
    Arrival_Time: str
    Price: str
    Link: str


class FlightList(BaseModel):
    flights: List[FlightFinderOutput]


class ActivityPlannerOutput(BaseModel):
    Food: str
    Activities: str

## Planning Agent

In [19]:
async def get_trip_divider(mcp_servers) -> Agent:
    instructions = f"""You are an trip planner. You will determine the number of days to stay in each city. 
    The number of days in each city should depend on how popular the city. For example, you will spend more days in London than in manchester
    as london offers more stuff to do.
    You will also determine the order in which cities are visited. The order of the cities in the list represents the order of the cities visted.
    Output a list where each element is the city and the number of days to spend in that city the start and end dates of the trip will be given).
    Do not put starting city in the output.
    Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
    """
    trip_divider = Agent(
        name="trip_divider",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
       output_type=TripDividerOutput,
    )
    return trip_divider

In [20]:
start_date = "2025-10-01"
end_date = "2025-10-04"
start_destination = "Waterloo, ON"
destinations = ["Nova Scotia"]
additional_info = ""

question = f"Detemrine the number of days to stay in each city and the order to visit them. Start from {start_destination} to {destinations[0]} on {start_date} and return on {end_date} with the following additional information: {additional_info}."

for server in mcp_servers:
    await server.connect()
trip_divider = await get_trip_divider(mcp_servers)
with trace("trip_divider"):
    result = await Runner.run(trip_divider, question, max_turns=30)
    print(result.final_output)

Destinations=[Destination(city='Halifax', days=2), Destination(city="Peggy's Cove", days=1)]


In [21]:
travel_schedule = []
for destination in result.final_output.Destinations:
    travel_schedule.append([destination.city, destination.days])

sample_travel_schedule = [["Halifax", 2], ["Lunenburg", 1]]

start = "Toronto"
start_date = "2025-10-01"
destinations = travel_schedule

# convert string to datetime object
date = datetime.strptime(start_date, "%Y-%m-%d")

flights = []
current = start

for city, stay_days in destinations:
    flights.append([current, city, date.strftime("%Y-%m-%d")])
    current = city
    date += timedelta(days=stay_days)  # move forward after staying

# return to start after last destination
flights.append([current, start, date.strftime("%Y-%m-%d")])

print(flights)

[['Toronto', 'Halifax', '2025-10-01'], ['Halifax', "Peggy's Cove", '2025-10-03'], ["Peggy's Cove", 'Toronto', '2025-10-04']]


## Flight Agent

In [22]:
async def get_flight_finder(mcp_servers) -> Agent:
    instructions = f"""You are a flight finder. You are able to search the web for flight details.
Based on the request, you carry out necessary research and respond with your findings.
The flight should be on the date provided and the start and end destination should be the ones provided.
Return a link to the booking website where the result was found
Give 1 flight option or max 2 flight options
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
If there isn't a specific request, then just respond with "NA
"""
    flight_finder = Agent(
        name="flight_finder",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
        output_type=FlightList,
    )
    return flight_finder

In [23]:
question = "flights from Toronto to Halifax on 2025-10-01"

for server in mcp_servers:
    await server.connect()
flight_finder = await get_flight_finder(mcp_servers)
with trace("flight_finder"):
    result = await Runner.run(flight_finder, question, max_turns=30)
    print(result.final_output)

flights=[FlightFinderOutput(Airline='Air Canada', Departure='Toronto (YYZ)', Arrival='Halifax (YHZ)', Departure_Time='2025-10-01', Arrival_Time='2025-10-01', Price='Check website for current prices', Link='https://www.aircanada.com/en-ca/flights-from-toronto-to-halifax'), FlightFinderOutput(Airline='Porter Airlines', Departure='Toronto (YYZ)', Arrival='Halifax (YHZ)', Departure_Time='2025-10-01', Arrival_Time='2025-10-01', Price='Check website for current prices', Link='https://www.flyporter.com/en_ca/flights-from-toronto-to-halifax')]


In [24]:
# assuming you already built this from earlier code
trip_plan = [
    ["Toronto", "Halifax", "2025-10-01"],
    ["Halifax", "Toronto", "2025-10-03"],
]

async def get_flights_for_trip(trip_plan, mcp_servers):
    all_results: List[FlightList] = []

    # connect once
    for server in mcp_servers:
        await server.connect()

    # get agent
    flight_finder = await get_flight_finder(mcp_servers)

    # loop through each leg
    for origin, destination, date in trip_plan:
        question = f"flights from {origin} to {destination} on {date}"
        with trace("flight_finder"):
            result = await Runner.run(flight_finder, question, max_turns=30)
            all_results.append(result.final_output)

    return all_results

# usage
all_flights = await get_flights_for_trip(trip_plan, mcp_servers)
print(all_flights)

Failed to parse JSONRPC message from server
Traceback (most recent call last):
  File "/Users/jaivaderaa/travel-agent/.venv/lib/python3.13/site-packages/mcp/client/stdio/__init__.py", line 155, in stdout_reader
    message = types.JSONRPCMessage.model_validate_json(line)
  File "/Users/jaivaderaa/travel-agent/.venv/lib/python3.13/site-packages/pydantic/main.py", line 746, in model_validate_json
    return cls.__pydantic_validator__.validate_json(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        json_data, strict=strict, context=context, by_alias=by_alias, by_name=by_name
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
pydantic_core._pydantic_core.ValidationError: 1 validation error for JSONRPCMessage
  Invalid JSON: EOF while parsing a value at line 1 column 0 [type=json_invalid, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/json_invalid
Failed to parse JSONRPC message f

[FlightList(flights=[FlightFinderOutput(Airline='Flair Airlines', Departure='Toronto Region of Waterloo Intl (YKF)', Arrival='Halifax Intl (YHZ)', Departure_Time='7:05 p.m. on 2025-10-01', Arrival_Time='9:15 p.m. on 2025-10-01', Price='C$160', Link='https://www.cheapflights.ca/flights-to-Halifax/Toronto/'), FlightFinderOutput(Airline='Flair Airlines', Departure='Toronto Pearson Intl (YYZ)', Arrival='Halifax Intl (YHZ)', Departure_Time='7:40 a.m. on 2025-10-01', Arrival_Time='9:50 a.m. on 2025-10-01', Price='Approximately C$153', Link='https://www.cheapflights.ca/flights-to-Halifax/Toronto/')]), FlightList(flights=[FlightFinderOutput(Airline='Air Canada', Departure='Halifax (YHZ)', Arrival='Toronto (YYZ)', Departure_Time='2025-10-03 (various times)', Arrival_Time='Various (check Air Canada site)', Price='Around CAD 150 (subject to change)', Link='https://www.aircanada.com/en-ca/flights-from-halifax-to-toronto'), FlightFinderOutput(Airline='Flair Airlines', Departure='Halifax (YHZ)', Arr

In [25]:
for i, flight_list in enumerate(all_flights, 1):
    print(f"--- Flight Option Set {i} ---")
    for flight in flight_list.flights:
        print(f"{flight.Airline}: {flight.Departure} → {flight.Arrival}")
        print(f"   Departure: {flight.Departure_Time}")
        print(f"   Arrival:   {flight.Arrival_Time}")
        print(f"   Price:     {flight.Price}")
        print(f"   Link:      {flight.Link}")
    print()

--- Flight Option Set 1 ---
Flair Airlines: Toronto Region of Waterloo Intl (YKF) → Halifax Intl (YHZ)
   Departure: 7:05 p.m. on 2025-10-01
   Arrival:   9:15 p.m. on 2025-10-01
   Price:     C$160
   Link:      https://www.cheapflights.ca/flights-to-Halifax/Toronto/
Flair Airlines: Toronto Pearson Intl (YYZ) → Halifax Intl (YHZ)
   Departure: 7:40 a.m. on 2025-10-01
   Arrival:   9:50 a.m. on 2025-10-01
   Price:     Approximately C$153
   Link:      https://www.cheapflights.ca/flights-to-Halifax/Toronto/

--- Flight Option Set 2 ---
Air Canada: Halifax (YHZ) → Toronto (YYZ)
   Departure: 2025-10-03 (various times)
   Arrival:   Various (check Air Canada site)
   Price:     Around CAD 150 (subject to change)
   Link:      https://www.aircanada.com/en-ca/flights-from-halifax-to-toronto
Flair Airlines: Halifax (YHZ) → Toronto (YYZ)
   Departure: 2025-10-03 (check site for details)
   Arrival:   Various
   Price:     From CAD 91 (subject to change)
   Link:      https://flights.flyflair

## Activity Agent

In [26]:
async def get_activity_planner(mcp_servers) -> Agent:
    instructions = f"""You are an activity planner. You are able to search the web for activities and food.
Based on the request, you carry out necessary research and respond with your findings.
The activities and food should be in the destination provided.
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
"""
    activity_planner = Agent(
        name="activity_planner",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
        output_type=ActivityPlannerOutput,
    )
    return activity_planner

In [27]:
question = "Plan things to do in Halifax for 3 days"

for server in mcp_servers:
    await server.connect()
activity_planner = await get_activity_planner(mcp_servers)
with trace("activity_planner"):
    result = await Runner.run(activity_planner, question, max_turns=30)
    print(result.final_output)

Food='Halifax offers a variety of great dining options, especially known for its fresh seafood including lobster. You can enjoy local dishes at waterfront restaurants as well as try craft beers in local breweries. Popular food spots often feature lobster, scallops, and other Atlantic Canadian specialties.' Activities="Day 1: Explore downtown Halifax including the waterfront, historic sites like the Citadel National Historic Site, and the Maritime Museum of the Atlantic. Enjoy a stroll along the harbor and take a harbor ferry for city views.\nDay 2: Visit Point Pleasant Park for hiking and ocean views followed by a visit to the Halifax Public Gardens. In the afternoon, explore the vibrant North End neighborhood with shops, cafes, and street art.\nDay 3: Take a day trip outside Halifax, such as to Peggy's Cove, a famous lighthouse and coastal village, or drive to nearby scenic locations and small towns for nature and culture experiences."


## Complete Trip Planning Function

In [28]:
async def plan_complete_trip(start_date, end_date, start_destination, destinations, additional_info=""):
    """
    Complete trip planning function that combines all agents
    """
    
    # Connect to MCP servers
    for server in mcp_servers:
        await server.connect()
    
    # 1. Plan trip divisions
    question = f"Determine the number of days to stay in each city and the order to visit them. Start from {start_destination} to {destinations[0]} on {start_date} and return on {end_date} with the following additional information: {additional_info}."
    
    trip_divider = await get_trip_divider(mcp_servers)
    with trace("trip_divider"):
        trip_result = await Runner.run(trip_divider, question, max_turns=30)
    
    # 2. Create flight schedule
    travel_schedule = [[dest.city, dest.days] for dest in trip_result.final_output.Destinations]
    
    date = datetime.strptime(start_date, "%Y-%m-%d")
    flight_schedule = []
    current = start_destination
    
    for city, stay_days in travel_schedule:
        flight_schedule.append([current, city, date.strftime("%Y-%m-%d")])
        current = city
        date += timedelta(days=stay_days)
    
    flight_schedule.append([current, start_destination, date.strftime("%Y-%m-%d")])
    
    # 3. Get flights
    all_flights = await get_flights_for_trip(flight_schedule, mcp_servers)
    
    # 4. Get activities for each destination
    activity_planner = await get_activity_planner(mcp_servers)
    activities_results = []
    
    for destination in trip_result.final_output.Destinations:
        question = f"Plan things to do in {destination.city} for {destination.days} days"
        with trace("activity_planner"):
            activity_result = await Runner.run(activity_planner, question, max_turns=30)
            activities_results.append({
                'city': destination.city,
                'days': destination.days,
                'activities': activity_result.final_output
            })
    
    return {
        'trip_plan': trip_result.final_output,
        'flight_schedule': flight_schedule,
        'flights': all_flights,
        'activities': activities_results
    }

In [29]:
def display_trip_plan(trip_data, start_date, end_date, start_destination):
    print("\n" + "="*60)
    print("           TRIP ITINERARY")
    print("="*60)

    print(f"\n📅 {start_date} to {end_date}")
    print(f"🏠 Starting from: {start_destination}\n")

    print("📍 DESTINATIONS:")
    for dest in trip_data['trip_plan'].Destinations:
        print(f"   • {dest.city}: {dest.days} days")

    print("\n✈️  FLIGHTS:")
    for i, (origin, dest, date) in enumerate(trip_data['flight_schedule'], 1):
        print(f"\n   Flight {i}: {origin} → {dest} ({date})")
        if i-1 < len(trip_data['flights']) and trip_data['flights'][i-1].flights:
            for flight in trip_data['flights'][i-1].flights:
                print(f"      • {flight.Airline}: {flight.Price}")
                print(f"        Departure: {flight.Departure_Time}")
                print(f"        Arrival: {flight.Arrival_Time}")
                print(f"        Link: {flight.Link}")

    print("\n🎯 ACTIVITIES:")
    for activity in trip_data['activities']:
        print(f"\n   {activity['city'].upper()} ({activity['days']} days)")
        print(f"   Food: {activity['activities'].Food}")
        print(f"   Activities: {activity['activities'].Activities}")
    
    print("\n" + "="*60)

## Dummy Data for Testing

In [ ]:
# Display the dummy trip plan
display_trip_plan(dummy_trip_data, start_date, end_date, start_destination)

In [ ]:
# Dummy trip data based on actual agent outputs
dummy_trip_data = {
    'trip_plan': TripDividerOutput(
        Destinations=[
            Destination(city='Halifax', days=2),
            Destination(city='Lunenburg', days=1)
        ]
    ),
    'flight_schedule': [
        ['Toronto', 'Halifax', '2025-10-01'],
        ['Halifax', 'Lunenburg', '2025-10-03'],
        ['Lunenburg', 'Toronto', '2025-10-04']
    ],
    'flights': [
        FlightList(flights=[
            FlightFinderOutput(
                Airline='Flair Airlines',
                Departure='Toronto (YYZ)',
                Arrival='Halifax (YHZ)',
                Departure_Time='7:40 a.m.',
                Arrival_Time='9:50 a.m.',
                Price='C$ 153',
                Link='https://www.cheapflights.ca/flights-to-Halifax/Toronto/'
            ),
            FlightFinderOutput(
                Airline='WestJet',
                Departure='Toronto (YYZ)',
                Arrival='Halifax (YHZ)',
                Departure_Time='10:30 p.m.',
                Arrival_Time='12:42 a.m. next day',
                Price='C$ 160',
                Link='https://www.cheapflights.ca/flights-to-Halifax/Toronto/'
            )
        ]),
        FlightList(flights=[]),  # No direct flights Halifax to Lunenburg
        FlightList(flights=[
            FlightFinderOutput(
                Airline='Air Canada',
                Departure='Halifax (YHZ)',
                Arrival='Toronto (YYZ)',
                Departure_Time='2025-10-04 10:30 AM',
                Arrival_Time='2025-10-04 12:45 PM',
                Price='CAD 154',
                Link='https://www.aircanada.com/en-ca/flights-to-toronto'
            )
        ])
    ],
    'activities': [
        {
            'city': 'Halifax',
            'days': 2,
            'activities': ActivityPlannerOutput(
                Food="Some top restaurants to try in Halifax include:\n- Salt + Ash Beach House: Known for delicious food and cold beer in a relaxed atmosphere.\n- Press Gang Restaurant: Famous for a wide variety of oysters and Atlantic seafood.\n- The Bicycle Thief: Italian and seafood dishes with waterfront views.\n- Chives Canadian Bistro: Local, seasonal dishes with a focus on seafood.",
                Activities="Day 1: Explore the Halifax Waterfront including the boardwalk, Pier 21, and historic sites. Visit Halifax Citadel National Historic Site. Stroll through Halifax Public Gardens. Evening: Listen to traditional Celtic music at a local pub.\n\nDay 2: Take a harbor kayak tour to see the city's skyline. Visit the Maritime Museum of the Atlantic. Ferry ride to Dartmouth for unique views and local cuisine."
            )
        },
        {
            'city': 'Lunenburg',
            'days': 1,
            'activities': ActivityPlannerOutput(
                Food='The Salt Shaker Deli, The Savvy Sailor, and The Grand Banker Bar & Grill are popular dining options in Lunenburg, offering local seafood and Nova Scotian cuisine.',
                Activities="Explore Lunenburg, a UNESCO World Heritage Site. Walk along the scenic waterfront and visit the Fisheries Museum of the Atlantic. Take a guided boat tour such as Sail Lunenburg Star Charters. Explore the colorful Old Town, see the historic Bluenose II schooner, and visit the Lunenburg Academy. Visit Blue Rocks, a quaint fishing village nearby known for its art and coastal beauty."
            )
        }
    ]
}

# Test configuration
start_date = "2025-10-01"
end_date = "2025-10-04"
start_destination = "Toronto"

print("Dummy data created successfully!")